# 08: QAT fine-tuning

Post-training ternary quantization (smelt.quantize) is fast but degrades quality.
QAT recovers accuracy by training with quantized weights in the loop.

| method | speed | quality | when to use |
|:---|:---|:---|:---|
| smelt.quantize | seconds | lower | already-ternary models (BitNet) |
| smelt.prepare_qat + train + smelt.freeze | hours (GPU) | higher | float models (Llama, Mistral) |

In [1]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path("../")))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import smelt

## Float model

TinyLlama 1.1B. Small enough to fine-tune in minutes on a single GPU.

In [2]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
model.eval()

print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

params: 1100M


### Float output

In [3]:
prompts = [
    "The capital of UK is",
    "def fibonacci(n):",
    "Explain x86 AVX2 in simple terms:",
]
ids = tokenizer.encode(prompts[0], return_tensors="pt")

for p in prompts:
    p_ids = tokenizer.encode(p, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(p_ids, max_new_tokens=32, do_sample=False)
    print(f"float: {tokenizer.decode(out[0], skip_special_tokens=True)}")
    print()

Both `max_new_tokens` (=32) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=32) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


float: The capital of UK is London.

2. The United States of America:

a. The capital of USA is Washington D.C.

b. The capital



Both `max_new_tokens` (=32) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


float: def fibonacci(n):
    if n == 0:
        return 0
    elif n == 1:
        return 1
    else:
       

float: Explain x86 AVX2 in simple terms: What is AVX2 and how does it differ from AVX?



## Post-training quantize (no fine-tuning)

Absmean scaling: $\gamma = \text{mean}(|W|)$, then $W_t = \text{round}(W / \gamma)$ clamped to {-1, 0, +1}.
Fast but every weight loses precision in one shot.

In [4]:
model_ptq = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
smelt.quantize(model_ptq)

with torch.no_grad():
    out = model_ptq.generate(ids, max_new_tokens=32, do_sample=False)
print(f"PTQ:   {tokenizer.decode(out[0], skip_special_tokens=True)}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=32) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PTQ:   The capital of UK is És És És És És És És És És És És És És És És És És És És És És És És És És És És És És És És És


## Straight-Through Estimator (STE)

Problem: ternary quantization (round + clamp) has zero gradient almost everywhere.
Can't backprop through it.

STE fix: pretend quantization is the identity function during backward pass.
Forward uses ternary weights, backward updates the float shadow as if no rounding happened.

```
w_float  ──[ quantize ]──> w_ternary ──[ matmul ]──> output
                                                        |
w_float  <────────────── gradient ◄─────────────────────┘
              (skip quantize, "straight-through")
```

In code: `w_ste = w_float + (w_ternary - w_float).detach()`.
Forward value is w_ternary. Gradient flows to w_float.

smelt.prepare_qat() replaces each nn.Linear with this STE wrapper.
smelt.freeze() converts back to TernaryLinear + packs TL1 + fits PLAC + int norms.

In [5]:
model_qat = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)

if torch.cuda.is_available():
    model_qat = model_qat.cuda()

smelt.prepare_qat(model_qat)
n_train = sum(p.numel() for p in model_qat.parameters() if p.requires_grad)
print(f"STE layers inserted, trainable params: {n_train / 1e6:.0f}M")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 44.00 MiB. GPU 0 has a total capacity of 3.63 GiB of which 17.81 MiB is free. Process 1558 has 67.89 MiB memory in use. Including non-PyTorch memory, this process has 3.54 GiB memory in use. Of the allocated memory 3.48 GiB is allocated by PyTorch, and 1.70 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:1000]")
print(f"training samples: {len(dataset)}")

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model_qat,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="/tmp/smelt_qat",
        num_train_epochs=1,
        learning_rate=1e-4,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        logging_steps=10,
        bf16=torch.cuda.is_available(),
        max_seq_length=256,
    ),
)

trainer.train()

In [ ]:
model_qat = model_qat.cpu()
smelt.freeze(model_qat)
print("frozen to ternary + PLAC + int norms")

with torch.no_grad():
    out = model_qat.generate(ids, max_new_tokens=32, do_sample=False)
print(f"QAT:   {tokenizer.decode(out[0], skip_special_tokens=True)}")

## Compare

In [ ]:
for p in prompts:
    p_ids = tokenizer.encode(p, return_tensors="pt")
    with torch.no_grad():
        out_ptq = model_ptq.generate(p_ids, max_new_tokens=32, do_sample=False)
        out_qat = model_qat.generate(p_ids, max_new_tokens=32, do_sample=False)
    print(f"prompt: {p}")
    print(f"  PTQ: {tokenizer.decode(out_ptq[0], skip_special_tokens=True)}")
    print(f"  QAT: {tokenizer.decode(out_qat[0], skip_special_tokens=True)}")
    print()